# Desafio Final - ClearBank

Análise de transações a partir do arquivo `transacoes.csv`.

Deixe o CSV na mesma pasta do notebook e execute todas as células.

In [ ]:
import csv
import json
from datetime import datetime, date
from collections import defaultdict

LIMITE_SUSPEITO = 10000.00
TIPOS_VALIDOS = {"credito", "debito"}
ARQUIVO_CSV = "transacoes.csv"
ARQUIVO_JSON = "relatorio.json"

## Leitura do CSV

In [ ]:
def ler_transacoes(caminho: str) -> list[dict]:
    try:
        with open(caminho, encoding="utf-8", newline="") as arquivo:
            leitor = csv.DictReader(arquivo)
            return list(leitor)
    except FileNotFoundError:
        print(f"Arquivo não encontrado: {caminho}")
        return []

## Validação

In [ ]:
def validar_id(texto: str | None) -> int | None:
    if texto is None:
        return None
    texto = str(texto).strip()
    if not texto:
        return None
    try:
        return int(texto)
    except ValueError:
        return None


def validar_valor(texto: str | None) -> float | None:
    if texto is None:
        return None
    try:
        valor = float(str(texto).strip())
    except (ValueError, AttributeError):
        return None
    if valor <= 0:
        return None
    return valor


def converter_data(texto: str | None) -> datetime | None:
    if texto is None:
        return None
    try:
        return datetime.strptime(str(texto).strip(), "%Y-%m-%d")
    except (ValueError, AttributeError):
        return None


def validar_transacao(linha: dict) -> dict | None:
    try:
        transacao_id = validar_id(linha.get("id"))
        cliente_id = str(linha.get("cliente_id") or "").strip()
        data_obj = converter_data(linha.get("data"))
        tipo = str(linha.get("tipo") or "").strip().lower()
        valor = validar_valor(linha.get("valor"))
    except (KeyError, AttributeError, TypeError):
        return None

    if transacao_id is None or not cliente_id or data_obj is None:
        return None
    if tipo not in TIPOS_VALIDOS or valor is None:
        return None

    return {
        "id": transacao_id,
        "data": data_obj,
        "mes": data_obj.strftime("%Y-%m"),
        "cliente_id": cliente_id,
        "tipo": tipo,
        "valor": valor,
        "descricao": str(linha.get("descricao") or "").strip(),
        "categoria": str(linha.get("categoria") or "").strip(),
    }


def limpar_transacoes(linhas: list[dict]) -> tuple[list[dict], int]:
    validas: list[dict] = []
    ids_vistos: set[int] = set()
    invalidas = 0

    for linha in linhas:
        registro = validar_transacao(linha)
        if registro is None or registro["id"] in ids_vistos:
            invalidas += 1
            continue
        ids_vistos.add(registro["id"])
        validas.append(registro)

    return validas, invalidas


def exibir_resumo_limpeza(total_lidas: int, validas: int, invalidas: int) -> None:
    print(f"Total de linhas lidas: {total_lidas}")
    print(f"Linhas válidas: {validas}")
    print(f"Linhas inválidas: {invalidas}")

## Métricas e relatório

In [ ]:
def calcular_periodo(transacoes: list[dict]) -> dict:
    if not transacoes:
        return {"inicio": None, "fim": None, "dias": 0}

    datas = [item["data"] for item in transacoes]
    inicio = min(datas)
    fim = max(datas)
    return {
        "inicio": inicio.strftime("%Y-%m-%d"),
        "fim": fim.strftime("%Y-%m-%d"),
        "dias": (fim - inicio).days,
    }


def agrupar_por_mes(transacoes: list[dict]) -> dict[str, list[dict]]:
    grupos: dict[str, list[dict]] = defaultdict(list)
    for item in transacoes:
        grupos[item["mes"]].append(item)
    return dict(sorted(grupos.items()))


def calcular_metricas_mes(lista: list[dict]) -> dict:
    valores = [item["valor"] for item in lista]
    total_credito = sum(item["valor"] for item in lista if item["tipo"] == "credito")
    total_debito = sum(item["valor"] for item in lista if item["tipo"] == "debito")
    quantidade = len(lista)
    media = (sum(valores) / quantidade) if quantidade else 0.0
    return {
        "quantidade": quantidade,
        "total_credito": round(total_credito, 2),
        "total_debito": round(total_debito, 2),
        "saldo": round(total_credito - total_debito, 2),
        "media": round(media, 2),
        "maior_valor": round(max(valores), 2) if valores else 0.0,
        "menor_valor": round(min(valores), 2) if valores else 0.0,
    }


def identificar_suspeitas(transacoes: list[dict]) -> list[dict]:
    suspeitas = []
    for item in transacoes:
        if item["valor"] > LIMITE_SUSPEITO:
            suspeitas.append({
                "id": item["id"],
                "cliente_id": item["cliente_id"],
                "data": item["data"].strftime("%Y-%m-%d"),
                "valor": round(item["valor"], 2),
            })
    return suspeitas


def gerar_relatorio(validas: list[dict], invalidas: int) -> dict:
    grupos = agrupar_por_mes(validas)
    resumo_mensal = {
        mes: calcular_metricas_mes(lista) for mes, lista in grupos.items()
    }
    return {
        "gerado_em": date.today().isoformat(),
        "total_transacoes_validas": len(validas),
        "total_transacoes_invalidas": invalidas,
        "periodo": calcular_periodo(validas),
        "resumo_mensal": resumo_mensal,
        "transacoes_suspeitas": identificar_suspeitas(validas),
    }


def salvar_json(relatorio: dict, caminho: str) -> bool:
    try:
        with open(caminho, "w", encoding="utf-8") as arquivo:
            json.dump(relatorio, arquivo, ensure_ascii=False, indent=2)
        return True
    except OSError as erro:
        print(f"Não foi possível salvar {caminho}: {erro}")
        return False


def formatar_moeda(valor: float) -> str:
    texto = f"{valor:,.2f}"
    return "R$ " + texto.replace(",", "X").replace(".", ",").replace("X", ".")


def exibir_relatorio(relatorio: dict) -> None:
    periodo = relatorio["periodo"]
    print("===== RELATÓRIO MENSAL =====")
    print()
    print(f"Gerado em: {relatorio['gerado_em']}")
    print(
        f"Período analisado: {periodo['inicio']} → {periodo['fim']} "
        f"({periodo['dias']} dias)"
    )
    print(f"Transações válidas: {relatorio['total_transacoes_validas']}")
    print(f"Transações inválidas: {relatorio['total_transacoes_invalidas']}")
    print()

    for mes, metricas in relatorio["resumo_mensal"].items():
        print(f"Mês: {mes}")
        print(f"  Transações: {metricas['quantidade']}")
        print(f"  Total crédito: {formatar_moeda(metricas['total_credito'])}")
        print(f"  Total débito:  {formatar_moeda(metricas['total_debito'])}")
        print(f"  Saldo:         {formatar_moeda(metricas['saldo'])}")
        print(f"  Média:         {formatar_moeda(metricas['media'])}")
        print(f"  Maior valor:   {formatar_moeda(metricas['maior_valor'])}")
        print(f"  Menor valor:   {formatar_moeda(metricas['menor_valor'])}")
        print()

    print("===== TRANSAÇÕES SUSPEITAS =====")
    suspeitas = relatorio["transacoes_suspeitas"]
    if not suspeitas:
        print("Nenhuma transação suspeita encontrada.")
        return
    for item in suspeitas:
        print(
            f"ID: {item['id']} | Cliente: {item['cliente_id']} | "
            f"Data: {item['data']} | Valor: {formatar_moeda(item['valor'])}"
        )

## Execução

In [ ]:
def main() -> None:
    linhas = ler_transacoes(ARQUIVO_CSV)
    validas, invalidas = limpar_transacoes(linhas)
    print("===== RESUMO DA LIMPEZA =====")
    exibir_resumo_limpeza(len(linhas), len(validas), invalidas)
    print()
    relatorio = gerar_relatorio(validas, invalidas)
    exibir_relatorio(relatorio)
    if salvar_json(relatorio, ARQUIVO_JSON):
        print()
        print(f"Relatório salvo em {ARQUIVO_JSON}")


main()